In [243]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import math
from collections import Counter

In [244]:
FEATURE_COLUMNS = ['beta_propensity', 'proline_fraction', 'net_charge', 'polar_fraction'] # 'AAT', 'TA', 'a3vSA' <- nie ma ich tu bo nie można obliczyć na bieżąco w treningu

path = "../files/df_ml_good_with_features.csv"
df = pd.read_csv(path)

print(df.shape)
df.head()

(1934, 21)


,id,sequence,length,class,merge_id,a3vSA,AAT,Na4vSS,THSA,nHS,...,net_charge,hydrophobicity,polar_fraction,nonpolar_fraction,acidic_fraction,basic_fraction,aromatic_fraction,beta_propensity,seq_entropy,proline_fraction
0,GI_171848907_PDB_2RNM_A__1_1,GRNSAKDIRTEERARVQLGNV,21,1,GI_171848907_PDB_2RNM_A__1_1_0,-0.457,0.190,-0.494,0.190,1.0,...,2.0,-1.185714,0.238095,0.380952,0.142857,0.238095,0.000000,0.953333,3.535175,0.000000
1,GI_342871650_GB_EGU74155_1_1,VRIYAKDIKSEEMARVRVGNE,21,1,GI_342871650_GB_EGU74155_1_1_0,-0.160,2.140,-0.165,1.871,1.0,...,1.0,-0.676190,0.142857,0.428571,0.190476,0.238095,0.047619,0.990000,3.427333,0.000000
2,GI_342887385_GB_EGU86897_1_1,GKNSAGRINGPGMVNIGNS,19,1,GI_342887385_GB_EGU86897_1_1_0,-0.256,1.438,-0.256,1.438,1.0,...,2.0,-0.563158,0.315789,0.578947,0.000000,0.105263,0.000000,0.937368,3.005315,0.052632
3,GI_347837243_EMB_CCD5181_1_1,HRIKIGKVTQASNAKAVIGVH,21,1,GI_347837243_EMB_CCD5181_1_1_0,-0.001,4.054,-0.008,4.054,2.0,...,4.2,-0.019048,0.190476,0.523810,0.000000,0.285714,0.000000,1.081429,3.296148,0.000000
4,GI_475677570_GB_EMT74561_1_1,VRNYASEIKGDEDAKVRLGND,21,1,GI_475677570_GB_EMT74561_1_1_0,-0.436,0.570,-0.435,0.000,0.0,...,-1.0,-1.138095,0.190476,0.380952,0.238095,0.190476,0.047619,0.912381,3.499228,0.000000


In [245]:
MAX_LEN = 50

FEATURE_COLUMNS = [
    'beta_propensity',
    'proline_fraction',
    'net_charge',
    'polar_fraction'
]

# scaling (IMPORTANT: before dataset)
scaler = StandardScaler()
df[FEATURE_COLUMNS] = scaler.fit_transform(df[FEATURE_COLUMNS])


AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
aa_to_idx = {aa: i+1 for i, aa in enumerate(AMINO_ACIDS)}
idx_to_aa = {i+1: aa for i, aa in enumerate(AMINO_ACIDS)}

PAD_IDX = 0
VOCAB_SIZE = len(AMINO_ACIDS) + 2
EOS = VOCAB_SIZE - 1


def encode_sequence(seq, max_len=MAX_LEN):
    seq = seq[:max_len]
    encoded = [aa_to_idx.get(a, PAD_IDX) for a in seq]
    encoded += [PAD_IDX] * (max_len - len(encoded))
    encoded.append(EOS)
    return torch.tensor(encoded, dtype=torch.long)


def decode_sequence(indices):
    seq = []
    for i in indices:
        if i == EOS:
            break
        if i != PAD_IDX:
            seq.append(idx_to_aa.get(i, ""))
    return "".join(seq)


class ProteinDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        seq = encode_sequence(row["sequence"])

        features = torch.tensor(
            row[FEATURE_COLUMNS].astype(float).to_numpy(),
            dtype=torch.float32
        )

        return seq, features

In [246]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [247]:
class ProteinCVAE(nn.Module):
    def __init__(self, vocab_size, feature_dim, d_model=128, latent_dim=64, max_len=50):
        super().__init__()

        self.latent_dim = latent_dim
        self.max_len = max_len
        self.d_model = d_model

        # ================= EMBEDDING =================
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)

        enc_layer = nn.TransformerEncoderLayer(d_model, 4, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, 2)

        # ================= FEATURES =================
        self.feature_mlp = nn.Sequential(
            nn.Linear(feature_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

        # ================= LATENT =================
        self.fc_mu = nn.Linear(d_model + 32, latent_dim)
        self.fc_logvar = nn.Linear(d_model + 32, latent_dim)

        # ================= DECODER CONDITIONING =================
        self.decoder_input = nn.Linear(latent_dim + 32, d_model)

        dec_layer = nn.TransformerDecoderLayer(d_model, 4, batch_first=True)
        self.decoder = nn.TransformerDecoder(dec_layer, 2)

        self.output = nn.Linear(d_model, vocab_size)

        self.pos = PositionalEncoding(d_model, max_len)

        # ================= FEATURE HEAD =================
        self.feature_predictor = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, feature_dim)
        )

    # ================= ENCODER =================
    def encode(self, x, f):
        x = self.embedding(x)
        x = self.pos(x)

        h_seq = self.encoder(x).mean(dim=1)
        f_enc = self.feature_mlp(f)

        h = torch.cat([h_seq, f_enc], dim=1)

        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)

        return mu, logvar, h_seq, f_enc

    # ================= REPARAM =================
    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    # ================= DECODER =================
    def decode(self, z, f_enc, tgt):
        z = torch.cat([z, f_enc], dim=1)

        memory = self.decoder_input(z).unsqueeze(1).repeat(1, tgt.size(1), 1)

        tgt = self.pos(self.embedding(tgt))

        out = self.decoder(tgt, memory)
        return self.output(out)

    # ================= FORWARD =================
    def forward(self, x, f):
        mu, logvar, h_seq, f_enc = self.encode(x, f)

        z = self.reparam(mu, logvar)

        logits = self.decode(z, f_enc, x)

        # STABILNIEJSZY FEATURE LOSS
        pred_features = self.feature_predictor(z)
        feature_loss = F.mse_loss(pred_features, f)

        return logits, mu, logvar, feature_loss

In [248]:
def loss_fn(logits, target, mu, logvar, feature_loss=None, beta=0.1, alpha=1.0):

    recon = F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        target.reshape(-1),
        ignore_index=0
    )

    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

    loss = recon + beta * kl

    if feature_loss is not None:
        loss = loss + alpha * feature_loss

    return loss, recon, kl, feature_loss

In [249]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ProteinCVAE(
    vocab_size=VOCAB_SIZE,
    feature_dim=len(FEATURE_COLUMNS),
    max_len=MAX_LEN
).to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

In [250]:
# split danych
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

# datasety
train_dataset = ProteinDataset(train_df)
val_dataset = ProteinDataset(val_df)

# loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    drop_last=False
)

In [251]:
best_val = float("inf")
patience = 5
counter = 0
best_state = None
EPOCHS = 60

for epoch in range(EPOCHS):

    #beta = min(1.0, epoch / 10)
    beta = min(0.3, epoch / 20)
    #beta = min(0.3, epoch / 30)

    # ===================== TRAIN =====================
    model.train()
    train_loss = 0

    for x, f in train_loader:
        x, f = x.to(device), f.to(device)

        inp = x[:, :-1]
        tgt = x[:, 1:]

        opt.zero_grad()

        logits, mu, logvar, feature_loss = model(inp, f)

        loss, recon, kl, feat = loss_fn(
            logits, tgt, mu, logvar,
            feature_loss=feature_loss,
            beta=beta,
            alpha=2.0
        )

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        opt.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ===================== VALIDATION =====================
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for x, f in val_loader:
            x, f = x.to(device), f.to(device)

            inp = x[:, :-1]
            tgt = x[:, 1:]

            logits, mu, logvar, feature_loss = model(inp, f)

            loss, _, _, _ = loss_fn(
                logits, tgt, mu, logvar,
                feature_loss=feature_loss,
                beta=beta,
                alpha=2.0
            )

            val_loss += loss.item()

    val_loss /= len(val_loader)

    #print(f"Epoch {epoch+1} | train {train_loss:.4f} | val {val_loss:.4f}")
    print(f"Epoch {epoch+1} | train {train_loss:.4f} | val {val_loss:.4f} | recon:{recon:.4f} | K:{kl:.3f} | F:{feature_loss:.3f}")

    # ===================== EARLY STOP =====================
    if val_loss < best_val - 1e-4:
        best_val = val_loss
        counter = 0
        best_state = model.state_dict()
        torch.save(best_state, "../best_cvae.pt")

    else:
        counter += 1

    if counter >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break


model.load_state_dict(best_state)
save_path = "files/models/cvae_protein_model1.pt"
torch.save(model.state_dict(), save_path)

Epoch 1 | train 4.5599 | val 3.5389 | recon:2.5109 | K:0.618 | F:0.190
Epoch 2 | train 3.1880 | val 2.5856 | recon:2.3953 | K:0.962 | F:0.072
Epoch 3 | train 2.4742 | val 2.1867 | recon:2.1667 | K:0.775 | F:0.056
Epoch 4 | train 1.9537 | val 1.0310 | recon:1.4052 | K:0.646 | F:0.018
Epoch 5 | train 1.0709 | val 0.3670 | recon:0.6627 | K:0.598 | F:0.026
Epoch 6 | train 0.6914 | val 0.3185 | recon:0.3719 | K:0.528 | F:0.047
Epoch 7 | train 0.5556 | val 0.3029 | recon:0.3459 | K:0.398 | F:0.070
Epoch 8 | train 0.4732 | val 0.2662 | recon:0.1704 | K:0.415 | F:0.055
Epoch 9 | train 0.4256 | val 0.2454 | recon:0.1383 | K:0.372 | F:0.029
Epoch 10 | train 0.3942 | val 0.2414 | recon:0.1247 | K:0.349 | F:0.048
Epoch 11 | train 0.3630 | val 0.2122 | recon:0.1403 | K:0.427 | F:0.037
Epoch 12 | train 0.3417 | val 0.2027 | recon:0.1903 | K:0.441 | F:0.026
Epoch 13 | train 0.3326 | val 0.1981 | recon:0.1018 | K:0.390 | F:0.031
Epoch 14 | train 0.3087 | val 0.2124 | recon:0.1687 | K:0.374 | F:0.088
E

In [255]:
def generate(model, features, scaler=None, max_len=50, device="cpu"):
    model.eval()

    with torch.no_grad():

        features = features.to(device)

        if scaler is not None:
            features = torch.tensor(
                scaler.transform(features.cpu().numpy().reshape(1, -1))[0],
                dtype=torch.float32,
                device=device
            )

        # feature encoder (POPRAWNIE)
        f_enc = model.feature_mlp(features).unsqueeze(0)

        # latent - LOSOWE
        z = torch.randn(1, model.latent_dim).to(device)

        seq = torch.zeros((1, max_len), dtype=torch.long).to(device)

        for i in range(max_len):

            logits = model.decode(z, f_enc, seq)   # <-- FIX HERE

            probs = F.softmax(logits[:, i], dim=-1)
            next_token = torch.multinomial(probs, 1)

            seq[:, i] = next_token

            if next_token.item() == EOS:
                break

        return decode_sequence(seq[0].cpu().numpy())

In [256]:
best_state = "files/models/cvae_protein_model.pt"
state_dict = torch.load(best_state, map_location="cpu")
model.load_state_dict(state_dict)

sample_features = scaler.transform(
    df[FEATURE_COLUMNS].iloc[10].values.reshape(1, -1)
)
sample_features = torch.tensor(sample_features[0], dtype=torch.float32)
generated_seq = generate(model, sample_features)
print("Generated:", generated_seq)
# Oryginalna sekwencja z pierwszego wiersza
original_seq = df["sequence"].iloc[10]
print("Original sequence:", original_seq)

# Wygenerowana sekwencja
sample_features = torch.tensor(df[FEATURE_COLUMNS].iloc[0].values, dtype=torch.float32)
generated_seq = generate(model, sample_features)
print("Generated sequence:", generated_seq)

Generated: EEEAA
Original sequence: TKQNIRDVKTTGNSIALAGLI
Generated sequence: PFRAL


C:\Users\marts\anaconda3\envs\protein310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [259]:
original_seq = df["sequence"].iloc[0]
print("Original sequence:", original_seq)

# wygenerowana sekwencja
sample_features = torch.tensor(df[FEATURE_COLUMNS].iloc[0].values, dtype=torch.float32)
generated_seq = generate(model, sample_features)
print("Generated sequence:", generated_seq)

# funkcja do cech (już była wcześniej)
def compute_features_from_sequence(seq):
    aa_list = list(seq)
    L = len(aa_list)

    kd = {'A':1.8,'C':2.5,'D':-3.5,'E':-3.5,'F':2.8,'G':-0.4,'H':-3.2,'I':4.5,
          'K':-3.9,'L':3.8,'M':1.9,'N':-3.5,'P':-1.6,'Q':-3.5,'R':-4.5,'S':-0.8,
          'T':-0.7,'V':4.2,'W':-0.9,'Y':-1.3}

    hydrophobicity = sum([kd.get(a,0) for a in aa_list])/L
    pos = sum(a in "KRH" for a in aa_list)
    neg = sum(a in "DE" for a in aa_list)
    charge = pos - neg
    from collections import Counter
    counts = Counter(aa_list)
    probs = torch.tensor([counts.get(a,0)/L for a in AMINO_ACIDS], dtype=torch.float32)
    entropy = -torch.sum(probs * torch.log(probs + 1e-8))
    aromatic_fraction = sum(a in "FWY" for a in aa_list)/L
    proline_fraction = sum(a=="P" for a in aa_list)/L
    pI = (pos + 1) / (neg + 1)
    instability_index = sum(a in "DEKPQR" for a in aa_list)/L

    return {
        "hydrophobicity": hydrophobicity,
        "charge": charge,
        "entropy": entropy.item(),
        "aromatic_fraction": aromatic_fraction,
        "proline_fraction": proline_fraction,
        "pI": pI,
        "instability_index": instability_index
    }

# 🔹 Obliczenie cech
orig_features = compute_features_from_sequence(original_seq)
gen_features = compute_features_from_sequence(generated_seq)

# 🔹 Porównanie w tabeli
import pandas as pd
comparison_df = pd.DataFrame([orig_features, gen_features], index=["Original", "Generated"])
print("\nFeature comparison:")
print(comparison_df)

Original sequence: GRNSAKDIRTEERARVQLGNV
Generated sequence: QVRAI

Feature comparison:
           hydrophobicity  charge   entropy  aromatic_fraction  \
Original        -1.185714       2  2.450396                0.0   
Generated        0.500000       1  1.609438                0.0   

           proline_fraction   pI  instability_index  
Original                0.0  1.5           0.428571  
Generated               0.0  2.0           0.400000  


z nielosowe

In [260]:
def generate_similar(model, seq, features, device="cpu"):
    model.eval()

    with torch.no_grad():
        seq = seq.unsqueeze(0).to(device)
        features = features.unsqueeze(0).to(device)

        mu, logvar, _, f_enc = model.encode(seq, features)

        z = mu + 0.1 * torch.randn_like(mu)

        out = torch.zeros((1, model.max_len), dtype=torch.long).to(device)

        for i in range(model.max_len):
            logits = model.decode(z, f_enc, out)
            probs = F.softmax(logits[:, i], dim=-1)

            token = torch.multinomial(probs, 1)
            out[:, i] = token

        return decode_sequence(out[0].cpu().numpy())

In [261]:
row = df.iloc[0]

seq = encode_sequence(row["sequence"])   # tensor [MAX_LEN+1]
features = torch.tensor(
    row[FEATURE_COLUMNS].values,
    dtype=torch.float32
)

features = torch.tensor(
    scaler.transform(row[FEATURE_COLUMNS].values.reshape(1, -1))[0],
    dtype=torch.float32
)

generated = generate_similar(
    model,
    seq,
    features,
    device=device
)

print("Original:", row["sequence"])
print("Generated:", generated)

TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.